# InformationTheory-01 — Entropie et information mutuelle

**Série InformationTheory** · Carnet 01 — Entropie et information mutuelle

La théorie de l'information de Shannon répond à une question précise : *combien
d'information* transporte une source, un canal, une observation ? Ce carnet fait
de cette question un **objet calculable** — l'entropie d'une distribution,
l'information qu'une variable porte sur une autre, le prix d'une mauvaise
distribution, la borne qu'impose un canal bruité — et le vérifie par le calcul,
sans GPU.


## Enjeu — mesurer l'information, pas l'intuition

« C'est informatif » est une impression. Shannon la remplace par un **nombre** :
l'**entropie**, la quantité moyenne de surprise d'une source. Tout le reste de la
théorie — ce qu'un canal transporte, ce qu'une observation apprend, combien on
peut compresser — se construit sur cette brique. On ne la postule pas : on la
**calcule** sur une distribution de probabilités.


## 0. L'entropie de Shannon

Pour une variable discrète `X` de loi `p`, l'entropie est

    H(X) = - sum_x  p(x) log2 p(x)        (en bits)

C'est l'incertitude moyenne avant d'observer `X`. On la calcule, et on regarde
comment elle réagit quand la distribution devient prévisible.


In [1]:
import numpy as np

def entropie_shannon(p, base=2.0):
    """Entropie de Shannon H(X) = -sum p log_b(p) ; ignore les p == 0."""
    p = np.asarray(p, dtype=float)
    p = p[p > 0]
    return float(-np.sum(p * np.log(p) / np.log(base)))

# Trois lois sur le meme support : juste, biaisee, determinee.
p_piere = [0.5, 0.5]
p_bias  = [0.9, 0.1]
p_det   = [1.0, 0.0]

print(f"Pièce juste        : H = {entropie_shannon(p_piere):.4f} bit")
print(f"Pionce biaisee (0.9): H = {entropie_shannon(p_bias):.4f} bit")
print(f"Deterministe       : H = {entropie_shannon(p_det):.4f} bit")
assert abs(entropie_shannon(p_piere) - 1.0) < 1e-9, "une piece juste a 1 bit d'entropie"
assert entropie_shannon(p_bias) < 1.0, "biaiser la piece reduit l'entropie"
assert entropie_shannon(p_det) == 0.0, "une variable determinee a entropie nulle"


Pièce juste        : H = 1.0000 bit
Pionce biaisee (0.9): H = 0.4690 bit
Deterministe       : H = -0.0000 bit


### Lecture du résultat

L'entropie est **maximale** quand tout est equally possible (la pièce juste,
`1` bit par faces) et **nulle** quand rien n'est à découvrir (la variable
déterminée). Baiser la probabilité d'un événement rare augmente un peu
l'incertitude moyenne ; rendre un résultat quasi-certain la fait chuter.
L'entropie mesure la **surprise moyenne**, pas le nombre d'issues.


## 1. L'entropie conditionnelle et l'information mutuelle

Deux variables `X` et `Y` partagent de l'information quand connaître `X` réduit
l'incertitude sur `Y`. Cette réduction est l'**information mutuelle** :

    I(X;Y) = H(Y) - H(Y|X) = H(X) + H(Y) - H(X,Y)

Elle est **symétrique** et **nulle** exactement quand `X` et `Y` sont
indépendants. On la calcule depuis la loi jointe.


In [2]:
def marges(joint):
    """Marginales (X lignes, Y colonnes) d'une jointe 2D."""
    joint = np.asarray(joint, dtype=float)
    return joint.sum(axis=1), joint.sum(axis=0)

def entropie_mutuelle(joint):
    """I(X;Y) = H(X) + H(Y) - H(X,Y), calculee sur la jointe."""
    px, py = marges(joint)
    hx, hy = entropie_shannon(px), entropie_shannon(py)
    j = np.asarray(joint, dtype=float); j = j[j > 0]
    hxy = float(-np.sum(j * np.log2(j)))
    return hx + hy - hxy

# Cas A : X et Y INDEPENDANTS (jointe = produit des marginales).
px = np.array([0.5, 0.5]); py = np.array([0.3, 0.7])
joint_ind = np.outer(px, py)

# Cas B : X et Y CORRELÉS (Y = X, sauf un faible bruit de bascule).
joint_dep = np.array([[0.48, 0.02],
                      [0.02, 0.48]])

print(f"Indépendants : I(X;Y) = {entropie_mutuelle(joint_ind):.4f} bit")
print(f"Correlés     : I(X;Y) = {entropie_mutuelle(joint_dep):.4f} bit")
assert abs(entropie_mutuelle(joint_ind)) < 1e-9, "indépendant -> information mutuelle nulle"
assert entropie_mutuelle(joint_dep) > 0.5, "corrélation -> information mutuelle positive"
print("=> I(X;Y) mesure exactement la réduction d'incertitude, et la détecte.")


Indépendants : I(X;Y) = 0.0000 bit
Correlés     : I(X;Y) = 0.7577 bit
=> I(X;Y) mesure exactement la réduction d'incertitude, et la détecte.


### Lecture du résultat

L'information mutuelle est le **pont** entre deux variables : nulle quand elles
ne se parlent pas, positive dès qu'il y a une liaison. Elle est symétrique
(`I(X;Y) = I(Y;X)`) — une propriété qui ne saute pas aux yeux mais que le calcul
donne pour rien. C'est la grandeur que le reste de la série (canaux, codage,
apprentissage) consomme.


## 2. La divergence de Kullback-Leibler

Supposons qu'on **code** selon une distribution `P` mais qu'on **décode** avec
une distribution `Q` (un modèle faux, une hypothèse erronée). Le surcoût moyen,
en bits, est la **divergence de Kullback-Leibler** :

    D_KL(P||Q) = sum_x p(x) log2 ( p(x) / q(x) )

Elle est **toujours positive** (inégalité de Gibbs) et nulle que si `P = Q`.
Elle relie aussi l'entropie croisée à l'entropie.


In [3]:
def divergence_kl(p, q):
    """D_KL(P||Q) = sum p log2(p/q), sommee sur le support de P."""
    p = np.asarray(p, dtype=float); q = np.asarray(q, dtype=float)
    m = p > 0
    return float(np.sum(p[m] * np.log2(p[m] / q[m])))

def entropie_croisee(p, q):
    """Entropie croisee H(P,Q) = -sum p log2 q."""
    p = np.asarray(p, dtype=float); q = np.asarray(q, dtype=float)
    m = p > 0
    return float(-np.sum(p[m] * np.log2(q[m])))

# P : la vraie loi ; Q : le modele (au large) utilise pour decoder.
p = np.array([0.7, 0.2, 0.1])
q = np.array([0.4, 0.4, 0.2])

h_p   = entropie_shannon(p)
kl    = divergence_kl(p, q)
h_cro = entropie_croisee(p, q)
print(f"H(P)              = {h_p:.4f} bit")
print(f"D_KL(P||Q)        = {kl:.4f} bit")
print(f"Entropie croisee  = {h_cro:.4f} bit")
assert kl > 0, "Gibbs : D_KL est strictement positive si P != Q"
assert abs(h_cro - (h_p + kl)) < 1e-9, "identite : entropie croisee = entropie + KL"
print("=> se tromper de distribution coute exactement D_KL bits par symbole.")


H(P)              = 1.1568 bit
D_KL(P||Q)        = 0.2651 bit
Entropie croisee  = 1.4219 bit
=> se tromper de distribution coute exactement D_KL bits par symbole.


### Lecture du résultat

`D_KL` est la **distance d'asymétrie** entre deux lois : le prix, en bits,
d'utiliser le mauvais modèle. Elle n'est pas une distance au sens classique
(non symétrique, `D_KL(P||Q) ≠ D_KL(Q||P)`) — c'est un **coût de décodage**.
L'identité `H(P,Q) = H(P) + D_KL(P||Q)` dit que l'entropie croisée est
l'entropie vraie **plus** la pénalité du faux modèle.


## 3. L'inégalité de traitement de l'information

Enchaînez deux canaux bruités : `X → Y → Z`. L'information que `Z` porte sur
`X` ne peut **pas excéder** celle que `Y` porte sur `X` :

    I(X;Z) ≤ I(X;Y)

C'est l'**inégalité de traitement de l'information** (data processing
inequality) : chaque maillon de la chaîne **efface** de l'information, jamais
n'en rajoute. On la vérifie sur une chaîne de deux canaux binaires.


In [4]:
def bsc(p_bascule):
    """Matrice de transition d'un canal binaire symetrique (bascule p)."""
    return np.array([[1.0-p_bascule, p_bascule],
                     [p_bascule, 1.0-p_bascule]])

px    = np.array([0.5, 0.5])
W_y   = bsc(0.1)      # X -> Y : 10 % de bascule
W_z   = bsc(0.3)      # Y -> Z : 30 % de bascule

joint_xy = np.diag(px) @ W_y                 # P(X,Y)
joint_xz = np.diag(px) @ (W_y @ W_z)         # P(X,Z) : Z ne voit X que via Y

i_xy = entropie_mutuelle(joint_xy)
i_xz = entropie_mutuelle(joint_xz)
print(f"I(X;Y) = {i_xy:.4f} bit    (un etage de bruit)")
print(f"I(X;Z) = {i_xz:.4f} bit    (deux etages de bruit)")
assert i_xz <= i_xy + 1e-9, "DPI : ajouter un etage ne fait qu'effacer de l'information"
print("=> chaque etage de bruit efface de l'information : I ne peut que decroitre.")


I(X;Y) = 0.5310 bit    (un etage de bruit)
I(X;Z) = 0.0752 bit    (deux etages de bruit)
=> chaque etage de bruit efface de l'information : I ne peut que decroitre.


### Lecture du résultat

C'est le **théorème du bruit** en une ligne : l'information ne se crée pas en
transit. Chacune des deux étapes de bruit **détruit** un peu de la liaison
initiale — `I(X;Z)` est strictement plus petite qu'`I(X;Y)`. C'est pourquoi
aucun préprocessement ni aucune mesure intermédiaire ne peut révéler ce que la
source n'a pas transmis.


## 4. La capacité d'un canal

Un canal **binaire symétrique** qui bascule chaque bit avec probabilité `δ` a
pour capacité

    C = 1 - H_b(δ)        (bit par symbole)

C'est le **débit fiable maximal** : au-delà, le bruit submerge le message. On la
calcule en fonction du taux de bascule.


In [5]:
def h_binaire(p):
    """Entropie binaire H_b(p) = H([p, 1-p])."""
    return entropie_shannon([p, 1.0 - p])

print(f"{'delta':>6}   {'capacite C = 1 - H_b(delta)':>26}")
caps = {}
for d in [0.0, 0.05, 0.1, 0.2, 0.5]:
    c = 1.0 - h_binaire(d)
    caps[d] = c
    print(f"{d:>6.2f}   {c:>26.4f} bit/symbole")
assert caps[0.0] == 1.0, "canal sans bruit : capacite = 1 bit"
assert caps[0.5] == 0.0, "canal purement aleatoire : capacite = 0"
assert caps[0.1] < caps[0.05] < caps[0.0], "plus le bruit est fort, plus la capacite chute"
print("\n=> la capacite est la borne superieure du debit FIABLE d'un canal.")


 delta   capacite C = 1 - H_b(delta)
  0.00                       1.0000 bit/symbole
  0.05                       0.7136 bit/symbole
  0.10                       0.5310 bit/symbole
  0.20                       0.2781 bit/symbole
  0.50                       0.0000 bit/symbole

=> la capacite est la borne superieure du debit FIABLE d'un canal.


## Synthèse — l'information comme objet calculé

| Brique | Ce qu'elle rend calculable |
|---|---|
| Entropie de Shannon | l'incertitude moyenne d'une source : `1` bit (juste) → `0` (déterminée) |
| Information mutuelle | ce qu'une variable révèle d'une autre : `0` (indépendant) → `>0` (corrélé) |
| Divergence KL | le surcoût, en bits, du mauvais modèle : toujours `≥ 0` |
| Inégalité DPI | l'information ne peut que décroître le long d'une chaîne bruitée |
| Capacité de canal | le débit fiable maximal : `1 - H_b(δ)`, de `1` (net) à `0` (saturé) |

La théorie de l'information n'est pas une intuition sur « l'informatif » : c'est
une **arithmétique des bits** qu'on calcule sur des distributions.


## Exercices


In [6]:
# Exercice 1 — Le theoreme du codage de canal
# Le carnet a donne la CAPACITE C d'un canal. Le theoreme de Shannon dit qu'il
# existe des codes atteignant arbitrairement C, mais seulement en BLOCS LONGS.
# Ecrire en quoi C est une borne en "regime asymptotique" (longueur de bloc
# tendant a l'infini) et pourquoi on ne l'atteint pas sur un seul symbole.
def theoreme_codage_canal():
    # TODO etudiant
    return None  # TODO etudiant


In [7]:
# Exercice 2 — L'entropie de taux (source avec memoire)
# H(X) mesure l'incertitude sur UN symbole. Si les symboles sont CORRELés
# (source avec memoire), l'entropie par symbole est plus petite que H(X) :
# c'est l'entropie de taux H_infini = lim_{n->inf} (1/n) H(X_1..X_n).
# Ecrire pourquoi les correlations font BAISSER l'entropie par symbole, et
# donner l'exemple d'une source a 2 etats (chaine de Markov) ou on calcule
# H_infini.
def entropie_de_taux():
    # TODO etudiant
    return None  # TODO etudiant


In [8]:
# Exercice 3 — L'entropie differentielle (variables continues)
# L'entropie de Shannon est definie pour les variables DISCRETES. Pour une
# variable CONTINUE on definit l'entropie differentielle h(x) = -int f log f.
# Ecrire pourquoi elle peut etre NEGATIVE (contrairement a H discrete) et
# pourquoi une densite tres pointe (f grande) donne une entropie differentielle
# tres negative : la "surprise" au sens de Shannon n'y est plus la meme.
def entropie_differentielle():
    # TODO etudiant
    return None  # TODO etudiant


## À retenir

- **L'entropie** `H(X) = -Σ p log p` est la surprise moyenne d'une source :
  maximale quand tout est possible, nulle quand tout est décidé.
- **L'information mutuelle** `I(X;Y) = H(X)+H(Y)-H(X,Y)` mesure ce que `X`
  révèle de `Y` ; nulle si et seulement si elles sont indépendantes.
- **La divergence KL** `D_KL(P||Q)` est le surcoût en bits du mauvais modèle ;
  toujours `≥ 0` (Gibbs), et `H(P,Q) = H(P) + D_KL(P||Q)`.
- **L'inégalité DPI** : l'information ne fait que décroître le long d'une chaîne
  `X→Y→Z` — le bruit efface, il ne crée pas.
- **La capacité** d'un canal binaire symétrique est `C = 1 - H_b(δ)`, le débit
  fiable maximal.


## Bibliographie

- C. E. Shannon, *A Mathematical Theory of Communication*, Bell System Technical
  Journal, 1948 — l'entropie et la capacité.
- T. M. Cover, J. A. Thomas, *Elements of Information Theory*, 2ᵉ éd., Wiley,
  2006 — le référentiel (entropie, KL, DPI, codage).
- Carnet 02 (à venir) : le codage de source — Huffman, la borne de Shannon.
